# Clause samples for risk scoring (#9)

Real annotated clause text from the CUAD training split, grouped by category, for designing the risk rules.

Scope is the 34 categories the classifier is trained on, minus the 6 that only identify the contract (document name, parties, dates, governing law), which leaves 28.

While reading each category, note:
- what separates a harmless instance from a risky one (wording, scope, duration, money)
- whether the clause binds one party or both
- red-flag phrases worth turning into rules

In [1]:
import textwrap
from pathlib import Path
import pandas as pd

## Load the prepared data

This works whether the notebook is launched from the repository root or from the `notebooks` directory.

In [2]:
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(path for path in candidates if (path / 'data/processed/contract_categories.jsonl').exists())

rows = pd.read_json(project_root / 'data/processed/contract_categories.jsonl', lines=True)
train = rows[rows['split'] == 'train']

descriptions = pd.read_csv(project_root / 'data/cuad/category_descriptions.csv')
descriptions = {
    cat.removeprefix('Category: ').strip().lower(): desc.removeprefix('Description: ').strip()
    for cat, desc in zip(descriptions.iloc[:, 0], descriptions.iloc[:, 1])
}

INFO_CATEGORIES = {'Document Name', 'Parties', 'Agreement Date', 'Effective Date', 'Expiration Date', 'Governing Law'}
categories = sorted(set(train['category']) - INFO_CATEGORIES)

spans = train[train['present']].explode('answers').dropna(subset=['answers'])
spans = spans.assign(text=spans['answers'].str['text'].str.strip())[['contract_id', 'category', 'text']]

print('Train contracts:', train['contract_id'].nunique())
print('Categories in scope:', len(categories))
print('Annotated spans in scope:', spans['category'].isin(categories).sum())

Train contracts: 408
Categories in scope: 28
Annotated spans in scope: 6967


## Overview

How often each category appears, and how long its clauses usually are.

In [3]:
n_contracts = train['contract_id'].nunique()
overview = (
    spans[spans['category'].isin(categories)]
    .groupby('category')
    .agg(contracts=('contract_id', 'nunique'), spans=('text', 'size'), median_chars=('text', lambda s: int(s.str.len().median())))
    .assign(pct_contracts=lambda d: (100 * d['contracts'] / n_contracts).round(1))
    .sort_values('contracts', ascending=False)
)
overview

,contracts,spans,median_chars,pct_contracts
category,,,,
Anti-Assignment,302,517,219,74.0
Cap On Liability,231,554,312,56.6
License Grant,205,642,358,50.2
Audit Rights,176,538,254,43.1
Termination For Convenience,154,205,160,37.7
Post-Termination Services,153,368,378,37.5
Renewal Term,150,179,273,36.8
Exclusivity,147,332,299,36.0
Insurance,134,443,248,32.8


## Helpers

`show` prints the CUAD description and random examples from different contracts. Change `seed` to see a different set.

`find` searches every span for a phrase, so you can check how common a red-flag phrase is and which categories use it.

In [4]:
def show(category, n=8, seed=0, max_chars=700):
    print(category.upper())
    print(textwrap.fill(descriptions.get(category.lower(), ''), 100))
    pool = spans[spans['category'] == category].drop_duplicates('contract_id')
    print(f'\n{len(pool)} contracts have this clause. Showing {min(n, len(pool))}.\n')
    for _, row in pool.sample(min(n, len(pool)), random_state=seed).iterrows():
        text = row['text'] if len(row['text']) <= max_chars else row['text'][:max_chars] + ' [...]'
        print('-', row['contract_id'])
        print(textwrap.indent(textwrap.fill(' '.join(text.split()), 100), '    '))
        print()


def find(phrase, category=None, n=5, seed=0):
    hits = spans[spans['category'].isin(categories) & spans['text'].str.contains(phrase, case=False, regex=False)]
    if category:
        hits = hits[hits['category'] == category]
    print(f'"{phrase}": {len(hits)} spans in {hits["contract_id"].nunique()} contracts\n')
    print(hits['category'].value_counts().to_string(), '\n')
    for _, row in hits.sample(min(n, len(hits)), random_state=seed).iterrows():
        print(f"- [{row['category']}] {row['contract_id']}")
        print(textwrap.indent(textwrap.fill(' '.join(row['text'].split())[:500], 100), '    '))
        print()

In [5]:
grouped = {'Liability and money': ['Uncapped Liability', 'Cap On Liability', 'Liquidated Damages', 'Insurance', 'Warranty Duration', 'Covenant Not To Sue'], "Restrictions on a party's business": ['Non-Compete', 'Exclusivity', 'No-Solicit Of Customers', 'No-Solicit Of Employees', 'Competitive Restriction Exception', 'Rofr/Rofo/Rofn'], 'Term and termination': ['Termination For Convenience', 'Change Of Control', 'Renewal Term', 'Notice Period To Terminate Renewal', 'Post-Termination Services'], 'IP and licensing': ['Ip Ownership Assignment', 'Joint Ip Ownership', 'License Grant', 'Non-Transferable License', 'Irrevocable Or Perpetual License', 'Affiliate License-Licensee', 'Anti-Assignment'], 'Commercial terms': ['Revenue/Profit Sharing', 'Minimum Commitment', 'Volume Restriction', 'Audit Rights']}
assert sorted(c for cats in grouped.values() for c in cats) == categories

## Liability and money

### Uncapped Liability

In [6]:
show('Uncapped Liability')

UNCAPPED LIABILITY
Is a party’s liability uncapped upon the breach of its obligation in the contract? This also
includes uncap liability for a particular type of breach such as IP infringement or breach of
confidentiality obligation.

98 contracts have this clause. Showing 8.

- FUSIONPHARMACEUTICALSINC_06_05_2020-EX-10.17-Supply Agreement - FUSION
    EXCEPT FOR DAMAGES FOR WHICH A PARTY IS RESPONSIBLE PURSUANT TO ITS INDEMNIFICATION OBLIGATIONS SET
    FORTH IN SECTION 10 ABOVE, EACH PARTY SPECIFICALLY DISCLAIMS ALL LIABILITY FOR AND SHALL IN NO EVENT
    BE LIABLE FOR ANY INCIDENTAL, SPECIAL, INDIRECT OR CONSEQUENTIAL DAMAGES, EXPENSES, LOST PROFITS,
    LOST SAVINGS, INTERRUPTIONS OF BUSINESS OR OTHER DAMAGES OF ANY KIND OR CHARACTER WHATSOEVER ARISING
    OUT OF OR RELATED TO THIS AGREEMENT OR RESULTING FROM THE MANUFACTURE, HANDLING. MARKETING, SALE,
    DISTRIBUTION OR USE OF LICENSED PRODUCT REGARDLESS OF THE FORM OF ACTION, WHETHER IN CONTRACT, TORT,
    STRICT LIABILITY OR OT

**Notes:** 

Uncapped liability seems pretty risky, but because of it's mutual benefit I'd say it's medium risk. The useful questions are who it applies to, what triggers it, and whether you’d be comfortable on the paying end of it. If the uncapped liability applies to you, that’s real exposure. One bad incident (a data leak, an IP lawsuit) could cost way more than the whole contract is worth, with no ceiling. If it applies to the other side, it’s actually protection for you, because it means they can’t hide behind a small cap if they seriously screw up. And since a lot of these clauses are mutual, you’re usually getting both the risk and the protection at once.

### Cap On Liability

In [ ]:
show('Cap On Liability')

CAP ON LIABILITY
Does the contract include a cap on liability upon the breach of a party’s obligation? This includes
time limitation for the counterparty to bring claims or maximum amount for recovery.

231 contracts have this clause. Showing 8.

- JOINTCORP_09_19_2014-EX-10.15-FRANCHISE AGREEMENT
    Except with respect to your obligations to indemnify us and claims that we may bring under Sections
    7, 9, 15, or 16 of this Agreement, and except for claims arising from your non-payment or
    underpayment of any amounts owed to us or our affiliates, (1) any and all claims arising out of or
    related to this Agreement or the relationship between you and us shall be barred, by express
    agreement of the parties, unless an action or proceeding is commenced within two (2) years from the
    date the cause of action accrues; and (2) you and we hereby waive to the fullest extent permitted by
    law, any right to or claim for any punitive or exemplary damages against the other, and ag

**Notes:** 

medium risk

### Liquidated Damages

In [8]:
show('Liquidated Damages')

LIQUIDATED DAMAGES
Does the contract contain a clause that would award either party liquidated damages for breach or a
fee upon the termination of a contract (termination fee)?

47 contracts have this clause. Showing 8.

- CORALGOLDRESOURCES,LTD_05_28_2020-EX-4.1-CONSULTING AGREEMENT
    This Agreement can be terminated at any time prior to the expiry of the Term, as follows:

- TRUENORTHENERGYCORP_02_08_2007-EX-10.1-DEVELOPMENT AGREEMENT
    By electing not to participate in any Completion attempt, or being deemed not to participate in any
    Completion attempt, Company shall be deemed to have relinquished to BP, and BP shall own and be
    entitled to receive, all of Company's interest in the Initial Well and share of production therefrom
    until the proceeds of the sale of such share, calculated at the well, or market value thereof if
    such share is not sold (after deducting applicable ad valorem, production, severance, windfall
    profits, and excise taxes, royalty, overridi

**Notes:** 

### Insurance

In [9]:
show('Insurance')

INSURANCE
Is there a requirement for insurance that must be maintained by one party for the benefit of the
counterparty?

134 contracts have this clause. Showing 8.

- TELKOMSALTD_01_30_2003-EX-10-LICENCE AND MAINTENANCE AGREEMENT
    Motor Vehicle Liability Insurance in respect of all motor vehicles brought onto the premises of
    TELKOM.

- ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement
    Each Party agrees to obtain and maintain, during the Term and for five (5) years after the Term,
    commercial general liability insurance, including products liability insurance, with minimum "A-" AM
    Best rated insurance carriers, in each case with limits of not less than five million dollars
    ($5,000,000) per occurrence and in the aggregate

- AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1
    Without prejudice to any term and condition under this Agreement, the Company shall maintain in
    force, at all times during the Term and [*****

**Notes:** 

### Warranty Duration

In [10]:
show('Warranty Duration')

WARRANTY DURATION
What is the duration of any  warranty against defects or errors in technology, products, or
services  provided under the contract?

65 contracts have this clause. Showing 8.

- NeuroboPharmaceuticalsInc_20190903_S-4_EX-10.36_11802165_EX-10.36_Manufacturing Agreement_ Supply Agreement
    Within [***] days after receipt of the Licensed Products and/or their matching placebo hereunder,
    NeuroBo may, in its discretion, perform a quality control test (the "Product Test") in accordance
    with the methods of the test on such Licensed Products and/or their matching placebo for acceptance
    (the "Product Test Methods"), which shall be separately agreed in writing by and between Dong-A and
    NeuroBo and attached hereto as Exhibit B, as may be amended by the Parties' agreement in writing
    from time to time.

- Magenta Therapeutics, Inc. - Master Development and Manufacturing Agreement
    In the event that Magenta does not inform Bachem within the [***] period that 

**Notes:** 

### Covenant Not To Sue

In [11]:
show('Covenant Not To Sue')

COVENANT NOT TO SUE
Is a party restricted from contesting the validity of the counterparty’s ownership of intellectual
property or otherwise bringing a claim against the counterparty for matters unrelated to the
contract?

76 contracts have this clause. Showing 8.

- ARMSTRONGFLOORING,INC_01_07_2019-EX-10.2-INTELLECTUAL PROPERTY AGREEMENT
    Without limitation to the foregoing, Arizona shall not file applications to register any Company
    Licensed IP or assist any person in doing the same, or contest, challenge, or otherwise take any
    action adverse to the Company's and its Affiliates' ownership of or rights in and to the Company
    Licensed IP, or assist any person in doing the same.

- ClickstreamCorp_20200330_1-A_EX1A-6 MAT CTRCT_12089935_EX1A-6 MAT CTRCT_Development Agreement
    The Developer recognizes the Client's right, title, and interest in and to all service marks,
    trademarks, trade names , Copyrights and Patents used by the Client and agrees not to engage in any


**Notes:** 

## Restrictions on a party's business

### Non-Compete

In [12]:
show('Non-Compete')

NON-COMPETE
Is there a restriction on the ability of a party to compete with the counterparty or operate in a
certain geography or business or technology sector?

96 contracts have this clause. Showing 8.

- EcoScienceSolutionsInc_20171117_8-K_EX-10.1_10956472_EX-10.1_Endorsement Agreement
    Talent represents and warrants that during the Term and in the Territories, Talent will not endorse
    or make any appearances or advertisements on behalf of any other product which is directly
    competitive to ESSI's products.

- PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4091580_EX-10.2_Content License Agreement_ Marketing Agreement_ Sales-Purchase Agreement2
    Supplier further agrees that it will not produce, cause to be produced or assist in the production
    of more units than are specified by Purchaser nor will Supplier produce, cause to be produced or
    assist in the production of any product or item not specifically requested by Purchaser using any or
    all of the Playboy Prop

**Notes:** 

### Exclusivity

In [13]:
show('Exclusivity')

EXCLUSIVITY
Is there an exclusive dealing  commitment with the counterparty? This includes a commitment to
procure all “requirements” from one party of certain technology, goods, or services or a prohibition
on licensing or selling technology, goods or services to third parties, or a prohibition on 
collaborating or working with other parties), whether during the contract or  after the contract
ends (or both).

147 contracts have this clause. Showing 8.

- LOYALTYPOINTINC_11_16_2004-EX-10.2-RESELLER AGREEMENT
    AEIS, on its own behalf, reserves the right to market its Cards and other products and services
    directly as 1 well as through additional firms on terms and conditions that it selects in its sole
    discretion, provided that no such sales shall take place within the NPO Marketplace.

- NEONSYSTEMSINC_03_01_1999-EX-10.5-DISTRIBUTOR AGREEMENT_Amendment
    Licensor and Licensee do hereby agree that this Amendment shall effect a change in the nature of the
    distributorship

**Notes:** 

### No-Solicit Of Customers

In [14]:
show('No-Solicit Of Customers')

NO-SOLICIT OF CUSTOMERS
Is a party restricted from contracting or soliciting customers or partners of the counterparty,
whether during the contract or after the contract ends (or both)?

27 contracts have this clause. Showing 8.

- GLOBALTECHNOLOGIESLTD_06_08_2020-EX-10.16-CONSULTING AGREEMENT
    he Company further agrees that neither it nor its employees, affiliates or assigns, shall enter
    into, or otherwise arrange (either for it/him/herself, or any other person or entity) any business
    relationship, contact any person regarding such Opportunity, either directly or indirectly, or any
    of its affiliates, or accept any compensation or advantage in relation to such Opportunity except as
    directly though Consultant, without the prior written approval of Consultant.

- MOELIS_CO_03_24_2014-EX-10.19-STRATEGIC ALLIANCE AGREEMENT
    Businesses involving Japanese Companies where all principal parties involved are located within
    Covered Regions

- BUFFALOWILDWINGSINC_06_05_1

**Notes:** 

### No-Solicit Of Employees

In [15]:
show('No-Solicit Of Employees')

NO-SOLICIT OF EMPLOYEES
Is there a restriction on a party’s soliciting or hiring employees and/or contractors from the 
counterparty, whether during the contract or after the contract ends (or both)?

49 contracts have this clause. Showing 8.

- MEDALISTDIVERSIFIEDREIT,INC_05_18_2020-EX-10.1-CONSULTING AGREEMENT
    CONSULTANT agrees that during its consultancy for REIT and for a period of twelve (12) months
    immediately following the termination of its consultancy with the Company for any reason, whether
    with or without cause, it will not, for itself or any other person or entity: (a) solicit, induce,
    recruit or encourage any of the Company's employees, contractors, independent contractors or any
    person who provides services to the Company to terminate or reduce their employment or other
    relationship with the Company; (b) hire any individual who is (or was within the six (6) months
    immediately preceding such hiring, unless such employee was terminated from such 

**Notes:** 

### Competitive Restriction Exception

In [16]:
show('Competitive Restriction Exception')

COMPETITIVE RESTRICTION EXCEPTION
This category includes the exceptions or carveouts to Non-Compete, Exclusivity and No-Solicit of
Customers above.

60 contracts have this clause. Showing 8.

- ASHWORTHINC_01_29_1999-EX-10.(D)-PROMOTION AGREEMENT AND NANTZ COMMUNICATIONS, INC.
    Notwithstanding the foregoing Nantz shall be permitted to wear a Lynx hat or clothing logo when
    performing promotional services for Lynx and to use Lynx equipment when performing any promotional
    services for the Company in which equipment will be used.

- XENCORINC_10_25_2013-EX-10.24-COLLABORATION AGREEMENT (3)
    The Parties acknowledge that nothing in this Agreement shall limit or restrict XENCOR, itself or
    with or through any third party, from developing and using any process (except for the Process) for
    the manufacture of any of its products, including the Product, provided that no BII Confidential
    Information and Know-How is used and XENCOR adheres to its confidentiality and non-use

**Notes:** 

### Rofr/Rofo/Rofn

In [17]:
show('Rofr/Rofo/Rofn')

ROFR/ROFO/ROFN
Is there a clause granting one party a right of first refusal, right of first offer or right of
first negotiation to purchase, license, market, or distribute equity interest, technology, assets,
products or services?

68 contracts have this clause. Showing 8.

- RemarkHoldingsInc_20081114_10-Q_EX-10.24_2895649_EX-10.24_Content License Agreement
    At the end of such [*] ([*]) year period, HSWI shall have the right to renew the Agreement under
    materially the same terms or shall have an option to purchase the Translated Content outright to the
    extent rights permit and to the extent the parties mutually agree to terms for such sale.

- FuelcellEnergyInc_20191106_8-K_EX-10.1_11868007_EX-10.1_Development Agreement
    During the Term of this Agreement and for two (2) years thereafter, in the event that either Party
    decides to sell or convey its interest in or otherwise dispose of any Prior JDA Project Patent to
    any Non-Affiliated Third Party, such Party will 

**Notes:** 

## Term and termination

### Termination For Convenience

In [18]:
show('Termination For Convenience')

TERMINATION FOR CONVENIENCE
Can a party terminate this  contract without cause (solely by giving a notice and allowing a
waiting  period to expire)?

154 contracts have this clause. Showing 8.

- BICYCLETHERAPEUTICSPLC_03_10_2020-EX-10.11-SERVICE AGREEMENT
    terminate your employment forthwith and to make a payment to you in lieu of salary and the benefits
    set out in paragraph 5 of this Agreement for all or any unexpired part of the notice period.

- Array BioPharma Inc. - LICENSE, DEVELOPMENT AND COMMERCIALIZATION AGREEMENT
    Commencing on the later of (i) [ * ] of the First Commercial Sale of the Binimetinib Product in the
    Ono Territory, or (ii) [ * ] of the First Commercial Sale of the Encorafenib Product in the Ono
    Territory, Ono shall have the right to terminate this Agreement, for any reasons by giving [ * ]
    advance written notice to Array which shall be accompanied by the rationale for such termination.

- MidwestEnergyEmissionsCorp_20080604_8-K_EX-10.2_30939

**Notes:** 

### Change Of Control

In [19]:
show('Change Of Control')

CHANGE OF CONTROL
Does one party have the right to terminate or is consent or notice required of the counterparty if
such party undergoes a change of control, such as a merger, stock sale, transfer of all or
substantially all of its assets or business, or assignment by operation of law?

95 contracts have this clause. Showing 8.

- ReynoldsConsumerProductsInc_20191115_S-1_EX-10.18_11896469_EX-10.18_Supply Agreement
    A change of control shall be deemed an assignment requiring consent hereunder provided that any
    transfer or assignment that results in Seller's and Buyer's current common parent, Reynolds Group
    Holdings Limited, ceasing to control either party shall not require consent of the other party.

- MACROGENICSINC_08_02_2013-EX-10-COLLABORATION AGREEMENT
    In the event of a Change in Control involving MacroGenics, MacroGenics shall provide prompt written
    notice to Green Cross following such Change in Control, and Green Cross may, in its sole discretion,
    termina

**Notes:** 

### Renewal Term

In [20]:
show('Renewal Term')

RENEWAL TERM
What is the renewal term after the initial term expires? This includes automatic extensions and
unilateral extensions with prior notice.

150 contracts have this clause. Showing 8.

- SYKESHEALTHPLANSERVICESINC_04_24_1998-EX-10.14-OUTSOURCING AGREEMENT
    Unless either party gives the other at least ninety days' prior written notice that it has elected
    not to extend the term of this Agreement beyond December 31, 1998, the term of this Agreement will
    be automatically extended until December 31, 1999.

- KUBIENT,INC_07_02_2020-EX-10.14-MASTER SERVICES AGREEMENT_Part1
    This Agreement will automatically renew for additional 1-year terms unless earlier terminated
    pursuant to this Agreement's express provisions (together with the Initial Term, such additional
    periods the "Term").

- BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement
    Thereafter, Bellicum shall have consecutive separate options to extend the Term for successive
    renewal term

**Notes:** 

### Notice Period To Terminate Renewal

In [21]:
show('Notice Period To Terminate Renewal')

NOTICE PERIOD TO TERMINATE RENEWAL
What is the notice period required to terminate renewal?

95 contracts have this clause. Showing 8.

- CORIOINC_07_20_2000-EX-10.5-LICENSE AND HOSTING AGREEMENT
    Thereafter, this Agreement shall automatically renew for subsequent one (1) year periods unless
    either party provides the other party with written notification at least thirty (30) days prior to
    the expiration of the initial five (5) year term or any one (1) year renewal thereof of its
    intention to terminate this Agreement.

- ENERGYXXILTD_05_08_2015-EX-10.13-Transportation AGREEMENT
    Subject to the other provisions of this Agreement, the term of this Agreement shall commence on the
    Effective Date and shall remain in effect until terminated by either Party upon thirty (30) days'
    prior written notice.

- RangeResourcesLouisianaInc_20150417_8-K_EX-10.5_9045501_EX-10.5_Transportation Agreement
    This Agreement shall commence on the Effective Date and continue in full 

**Notes:** 

### Post-Termination Services

In [22]:
show('Post-Termination Services')

POST-TERMINATION SERVICES
Is a party subject to obligations after the termination or expiration of a contract, including any
post-termination transition, payment, transfer of IP, wind-down, last-buy, or similar commitments?

153 contracts have this clause. Showing 8.

- TELEGLOBEINTERNATIONALHOLDINGSLTD_03_29_2004-EX-10.10-CONSTRUCTION AND MAINTENANCE AGREEMENT
    The remaining Parties to this Agreement shall assume the obligations, capital, operation, and
    maintenance interests of the Party terminating its participation in proportion to their interests
    assigned immediately preceding such effective date of termination, except for the continuing rights
    and obligations of the terminating Party as specified in Subparagraph 18.7 of this Agreement.

- MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSOURCING AGREEMENT
    Upon termination by IBM due to a default by MSL pursuant to Section 5.1 above, IBM shall, at its
    option, elect to do one of the following:

- Sonos, Inc. - 

**Notes:** 

## IP and licensing

### Ip Ownership Assignment

In [23]:
show('Ip Ownership Assignment')

IP OWNERSHIP ASSIGNMENT
Does intellectual property created  by one party become the property of the counterparty, either per
the terms of the contract or upon the occurrence of certain events?

101 contracts have this clause. Showing 8.

- MIDDLEBROOKPHARMACEUTICALS,INC_03_18_2010-EX-10.1-PROMOTION AGREEMENT
    In the event DD retains the service of a third party to perform any of DD's obligations hereunder DD
    shall, prior to commencement of any work by such third party, obtain the third party's written
    acknowledgement that all work done by such third party shall be deemed "work made for hire" and that
    the copyright in such material shall rest and remain with MBRK, or secure from such third party
    written assignment of all right, title and interest in and to the copyright in any material created
    by such third party.

- BERKELEYLIGHTS,INC_06_26_2020-EX-10.12-COLLABORATION AGREEMENT
    In no event shall Ginkgo, and Ginkgo shall cause its Affiliates to not, file any p

**Notes:** 

### Joint Ip Ownership

In [24]:
show('Joint Ip Ownership')

JOINT IP OWNERSHIP
Is there any clause providing for joint or shared ownership of intellectual property between the
parties to the contract?

39 contracts have this clause. Showing 8.

- INKTOMICORP_06_08_1998-EX-10.14-SOFTWARE HOSTING AGREEMENT
    All Usage Data shall be owned jointly by Microsoft and Inktomi, and Inktomi hereby irrevocably
    assigns to Microsoft an [*] interest therein.

- TheglobeComInc_19990503_S-1A_EX-10.20_5416126_EX-10.20_Co-Branding Agreement
    If the Domain Name is deemed a combination mark, neither party shall use the Domain Name for any
    purpose except as expressly provided herein or attempt to register the Domain Name, and the parties
    will jointly cooperate on any enforcement action of infringement of the Domain Name.

- BIOAMBERINC_04_10_2013-EX-10.34-DEVELOPMENT AGREEMENT (1)
    In the event it is not clear as to ownership of any Improvement as described in this Section 5.2, in
    other words, if it is not clear whether an invention or disco

**Notes:** 

### License Grant

In [25]:
show('License Grant')

LICENSE GRANT
Does the contract contain a license granted by one party to its counterparty?

205 contracts have this clause. Showing 8.

- ArcGroupInc_20171211_8-K_EX-10.1_10976103_EX-10.1_Sponsorship Agreement
    Subject to the terms and conditions of this Agreement, as part of the consideration of the full and
    timely payment of the Sponsor Fees, Club hereby grants to Sponsor, and Sponsor hereby accepts,
    solely in the Territory, and during the Term: (i) the right to use the Benefits set forth on Exhibit
    A and the license and right to use the Team Marks solely in connection with the advertisement and
    promotion of Sponsor's Dick's Wings and Grill branded restaurants (the "Sponsor Business") in
    accordance with this Agreement; and (ii) the right to use the designation "Official Wings of the
    Jacksonville Jaguars" and such other designations as Club and Sponsor may agree to in a writing from
    time to time ( [...]

- NlsPharmaceuticsLtd_20200228_F-1_EX-10.14_12029

**Notes:** 

### Non-Transferable License

In [26]:
show('Non-Transferable License')

NON-TRANSFERABLE LICENSE
Does the contract limit the ability of a party to transfer the license being granted to a third
party?



116 contracts have this clause. Showing 8.

- CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement
    Chase grants Affiliate a non-exclusive, nontransferable, revocable right to (a) access the Chase
    site through the links solely in accordance with the terms of this Agreement and (b) solely in
    connection with such links, to use Chase's logos, trade names, trademarks, and similar identifying
    material relating to Chase (collectively, the "Licensed Materials"), for the sole purpose of booking
    Chase products.

- SmartRxSystemsInc_20180914_1-A_EX1A-6 MAT CTRCT_11351705_EX1A-6 MAT CTRCT_Distributor Agreement
    For the term of this Agreement, Company grants Distributor a limited, revocable, non-transferable,
    non-exclusive license under Company's copyrights to use the Company Products at Distributor's
    facilities in the Territory solely for marketing and support purposes directly related to the
    performance of its duties under this Agreement.

**Notes:** 

### Irrevocable Or Perpetual License

In [27]:
show('Irrevocable Or Perpetual License')

IRREVOCABLE OR PERPETUAL LICENSE
Does the contract contain a  license grant that is irrevocable or perpetual?

57 contracts have this clause. Showing 8.

- Microgenics Corporation - Collaborative Development and Commercialization Agreement
    Achaogen shall be responsible for the prosecution and maintenance of any Patent applications and
    Patents claiming or covering any Achaogen Inventions, and Microgenics shall be responsible for the
    prosecution and maintenance of any Patent applications and Patents claiming or covering any
    Microgenics Inventions; provided, that [***] shall not, without first obtaining [***] prior written
    consent, file any Patent claiming or covering the [***]; provided, further, that, in the event that
    any such Patent applications covering or claiming any [***] are filed without first obtaining [***]
    prior written consent, then [***] hereby grants [***] a perpetual, irrevocable, fully paid-up,
    royalty-free, [...]

- LifewayFoodsInc_201603

**Notes:** 

### Affiliate License-Licensee

In [28]:
show('Affiliate License-Licensee')

AFFILIATE LICENSE-LICENSEE
Does the contract contain a license grant to a licensee (incl. sublicensor) and the affiliates of
such licensee/sublicensor?

47 contracts have this clause. Showing 8.

- RevolutionMedicinesInc_20200117_S-1_EX-10.1_11948417_EX-10.1_Development Agreement
    Subject to the terms and conditions of this Agreement, Sanofi hereby grants to RevMed a non-
    exclusive, royalty-free sublicense (which shall only be further sub-licensable (a) to RevMed's
    Subsidiaries, (b) to the Permitted Contractors or Researchers, and (c) solely with Sanofi's prior
    written consent, such consent not to be unreasonably withheld, delayed or conditioned, to Third
    Parties who are not Permitted Contractors or Researchers) under the rights exclusively licensed to
    Sanofi pursuant to Section 3.1, solely to the extent necessary for RevMed to perform its obligations
    under this Agreement and the Ancillary Agreements.

- KNOWLABS,INC_08_15_2005-EX-10-INTELLECTUAL PROPERTY AGR

**Notes:** 

### Anti-Assignment

In [29]:
show('Anti-Assignment')

ANTI-ASSIGNMENT
Is consent or notice required of a party if the contract is assigned to a third party?

302 contracts have this clause. Showing 8.

- LejuHoldingsLtd_20140121_DRS (on F-1)_EX-10.26_8473102_EX-10.26_Content License Agreement1
    This Agreement and any rights or authority granted hereunder shall not be assigned or transferred by
    either Party, including by operation of law, merger or otherwise, without the express written
    consent of the other Party, provided that Licensor may assign this Agreement without consent to any
    of its Affiliates and Licensee may assign this Agreement without consent to SINA Leju or an
    Affiliate of Licensee that is controlled by SINA Leju.

- ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement
    This Agreement may not be assigned or otherwise transferred, nor may any right or obligation
    hereunder be assigned or transferred, by either Party without the prior written consent of the other
    Party.

- Cy

**Notes:** 

## Commercial terms

### Revenue/Profit Sharing

In [30]:
show('Revenue/Profit Sharing')

REVENUE/PROFIT SHARING
Is one party required to share revenue or profit with the counterparty for any technology, goods,
or services?

131 contracts have this clause. Showing 8.

- CytodynInc_20200109_10-Q_EX-10.5_11941634_EX-10.5_License Agreement
    Vyera shall pay to CytoDyn royalties equal to fifty percent (50%) of Net Sales of Licensed Products
    in the Territory during the Royalty Term; provided that, after the Step-Down Date, the royalty
    percentage will be reduced to [***] of Net Sales of Licensed Products in the Territory throughout
    the remaining period in the Royalty Term.

- ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement
    Such royalty payment shall be payable to Pfizer within thirty (30) days of the end of each Calendar
    Quarter.

- VISIUMTECHNOLOGIES,INC_10_20_2004-EX-10.20-DISTRIBUTOR AGREEMENT
    Power2Ship will pay WLI 10% of any activation commissions ("Unit Commissions"), if any, it receives
    as a result of any Unit acti

**Notes:** 

### Minimum Commitment

In [31]:
show('Minimum Commitment')

MINIMUM COMMITMENT
Is there a minimum order size or minimum amount or units per-time period that one party must buy
from the counterparty under the contract?



133 contracts have this clause. Showing 8.

- BEYONDCOMCORP_08_03_2000-EX-10.2-CO-HOSTING AGREEMENT
    "Minimum Revenue Targets" shall mean: (i) in the first (1st) year of the Term, Aggregate Revenues of
    not less than Nine Million Dollars ($9,000,000) and (ii) in the second (2nd) year of the Term,
    Aggregate Revenues of not less than Twelve Million Dollars ($12,000,000).

- BLACKBOXSTOCKSINC_08_05_2014-EX-10.1-DISTRIBUTOR AGREEMENT
    *Minimum order of 1,000 Units

- ATMOSENERGYCORP_11_22_2002-EX-10.17-TRANSPORTATION SERVICE AGREEMENT
    Quantity of capacity to be released: Max 13,370 Dfli/Day, Min 13,370 DthDay.

- ENERGOUSCORP_03_16_2017-EX-10.24-STRATEGIC ALLIANCE AGREEMENT
    DIALOG may terminate this Agreement, immediately upon issuance of written notice to ENERGOUS in the
    event that: (A) DIALOG or its Affiliates fail to achieve a design-win pipeline with an annual
    projected sales value to DIALOG of at least [***] in the [***] after the availability of a Mass
 

**Notes:** 

### Volume Restriction

In [32]:
show('Volume Restriction')

VOLUME RESTRICTION
Is there a fee increase or consent requirement, etc. if one party’s use of the product/services
exceeds certain threshold?

65 contracts have this clause. Showing 8.

- FLOTEKINDUSTRIESINCCN_05_09_2019-EX-10.1-SUPPLY AGREEMENT
    "Maximum Quantity" means [***] pounds of Terpene Product per Year, prorated for any partial Year
    included in the Term.

- LeadersonlineInc_20000427_S-1A_EX-10.8_4991089_EX-10.8_Co-Branding Agreement
    In addition, as part of the purchase commitment set forth in Section 3.1 [Purchase Commitment],
    VerticalNet shall provide LeadersOnline up to $*** worth of Banner placements on the Home Page of
    other VerticalNet Sites of LeadersOnline's choice.

- GentechHoldingsInc_20190808_1-A_EX1A-6 MAT CTRCT_11776814_EX1A-6 MAT CTRCT_Distributor Agreement
    In addition to any other responsibilities stated in this Agreement, Company will: (a) Provide, at
    Distributor's reasonable request and without charge, up to 10 hours of training with

**Notes:** 

### Audit Rights

In [33]:
show('Audit Rights')

AUDIT RIGHTS


Does a party have the right to  audit the books, records, or physical locations of the counterparty
to ensure compliance with the contract?

176 contracts have this clause. Showing 8.

- NETGEAR,INC_04_21_2003-EX-10.16-DISTRIBUTOR AGREEMENT
    NETGEAR or its representative, at NETGEAR's cost may review these records during normal business
    hours for the sole purpose of determining Distributor's compliance with this Agreement.

- StampscomInc_20001114_10-Q_EX-10.47_2631630_EX-10.47_Co-Branding Agreement
    MBE will also permit the Company to enter any of MBE's premises during regular business hours to
    inspect the use of the Service in any reasonable manner.

- GSVINC_05_15_1998-EX-10-SPONSORSHIP AGREEMENT
    In addition, Excite may, upon no less than thirty (30) days prior written notice to Client, cause an
    independent Certified Public Accountant to inspect the records of Client reasonably related to the
    calculation of such payments during Client's normal business hou

**Notes:** 

## Red-flag phrase search

Try phrases you notice while reading. Some starting points:

In [34]:
find('sole discretion')

"sole discretion": 75 spans in 53 contracts

category
Termination For Convenience           11
Anti-Assignment                       10
Rofr/Rofo/Rofn                         7
Minimum Commitment                     6
Post-Termination Services              6
Change Of Control                      6
License Grant                          5
Affiliate License-Licensee             4
Exclusivity                            3
Competitive Restriction Exception      3
Renewal Term                           3
No-Solicit Of Employees                2
Non-Transferable License               2
Notice Period To Terminate Renewal     1
Insurance                              1
Ip Ownership Assignment                1
Revenue/Profit Sharing                 1
Joint Ip Ownership                     1
Non-Compete                            1
Audit Rights                           1 

- [Anti-Assignment] InnerscopeHearingTechnologiesInc_20181109_8-K_EX-10.6_11419704_EX-10.6_Distributor Agreement
    Distrib

In [35]:
find('without cause')

"without cause": 45 spans in 29 contracts



category
Termination For Convenience           29
Post-Termination Services              3
Notice Period To Terminate Renewal     2
No-Solicit Of Employees                2
Non-Compete                            2
Revenue/Profit Sharing                 2
Liquidated Damages                     2
Rofr/Rofo/Rofn                         1
No-Solicit Of Customers                1
Change Of Control                      1 

- [Post-Termination Services] ExactSciencesCorp_20180822_8-K_EX-10.1_11331629_EX-10.1_Promotion Agreement
    After the expiration of the Term or termination pursuant to Section 8.4 by either Party or Section
    8.6 by Exact, based on cumulative Incremental Laboratory Services Revenue achieved during the Term
    or up to the termination date, Exact agrees to pay Pfizer the applicable royalty payment set forth
    below for twelve (12) consecutive Calendar Quarters following the expiration of the Term (the "Tail
    Period"); provided, however, the Tail Period shall be r

In [36]:
find('perpetual')

"perpetual": 246 spans in 50 contracts

category
Irrevocable Or Perpetual License    99
License Grant                       91
Affiliate License-Licensee          22
Non-Transferable License            17
Exclusivity                          9
Post-Termination Services            3
Ip Ownership Assignment              2
Renewal Term                         1
Joint Ip Ownership                   1
Rofr/Rofo/Rofn                       1 

- [Affiliate License-Licensee] ZogenixInc_20190509_10-Q_EX-10.2_11663313_EX-10.2_Distributor Agreement
    Distributor hereby grants Zogenix an irrevocable, perpetual, royalty-free, fully paid-up, exclusive
    license with the right to grant sublicenses to use such Data solely generated and co-owned by
    Distributor outside of the Territory and a co-exclusive license in the Territory upon expiration or
    termination of the Agreement.

- [Irrevocable Or Perpetual License] FuelcellEnergyInc_20191106_8-K_EX-10.1_11868007_EX-10.1_Development Agreement


In [37]:
find('unlimited')

"unlimited": 27 spans in 12 contracts



category
License Grant                       9
Irrevocable Or Perpetual License    5
Non-Transferable License            4
Exclusivity                         2
Renewal Term                        2
Affiliate License-Licensee          2
Ip Ownership Assignment             1
Audit Rights                        1
Volume Restriction                  1 

- [License Grant] CORIOINC_07_20_2000-EX-10.5-LICENSE AND HOSTING AGREEMENT
    Subject to the terms and conditions of this Agreement, Commerce One hereby grants to Corio a fee-
    bearing, perpetual and irrevocable, nonexclusive, nontransferable (except in accordance with Section
    14.1 of this Agreement), right and license in the Territory to (i) reproduce the Software in machine
    executable object code format only for installation on the Corio Servers; (ii) install multiple
    copies of the Software on Corio's Servers which will be made remotely accessible to Corio's
    Customers for

- [License Grant] BIOAMBERINC_04_10_2013-EX